## Match Descriptors Between Two Views

# Feature Matching and Descriptor Alignment

Welcome to Unit 4 of our course. Up to this point, you have learned how to preprocess images and extract visual fingerprints by detecting keypoints and computing their descriptors.

Now, we are ready to tackle the first real dependency in our image stitching pipeline: **feature matching**. Finding keypoints in an isolated image is a great start, but to stitch a panorama, we must find the exact same points across two different views. By matching these descriptors, we create the critical links needed to eventually align and stitch images.

In this lesson, we will build a reliable feature matcher using OpenCV. We will configure our matcher based on descriptor data type and use Lowe's Ratio Test, a powerful technique for filtering out ambiguous matches.

---

## Handling Different Descriptor Types

Different algorithms produce different descriptor formats. SIFT produces arrays of floating-point numbers. ORB and AKAZE produce compact binary descriptors, represented as 8-bit unsigned integers.

Because our pipeline supports all three methods, our matcher needs to inspect the descriptor type before choosing a matching strategy.

```python
import numpy as np

def descriptors_are_binary(descriptors):
    return descriptors is not None and descriptors.dtype == np.uint8

```

If the descriptor `dtype` is `np.uint8`, we treat it as binary. Otherwise, we treat it as a floating-point descriptor such as SIFT.

---

## Lowe's Ratio Test

When matching descriptors, the algorithm computes distances between numeric fingerprints. A shorter distance means two descriptors look more similar.

For each descriptor in the first image, we ask for the two nearest candidates in the second image. The best candidate should be clearly better than the runner-up. If the two candidates are too close in quality, the match is ambiguous and should be rejected.

**Clear Match Example:**

```text
Descriptor from image A
       |
       |-- best candidate in image B      distance = 30
       |-- second-best candidate in B     distance = 80

Ratio check: best distance < ratio * second-best distance
With ratio = 0.75:
30 < 0.75 * 80
30 < 60  -> keep the match

```

**Ambiguous Match Example:**

```text
Descriptor from image A
       |
       |-- best candidate in image B      distance = 55
       |-- second-best candidate in B     distance = 60

With ratio = 0.75:
55 < 0.75 * 60
55 < 45  -> reject the match

```

The ratio acts like a strictness dial:

* **Lower values** (such as `0.65`) are stricter and produce fewer matches.
* **Higher values** (such as `0.85`) are looser and may keep more false matches.

The core filtering loop is short:

```python
good = []
for pair in raw_matches:
    if len(pair) == 2:
        best, second = pair
        if best.distance < ratio * second.distance:
            good.append(best)

```

---

## Understanding the FLANN Matcher

When we match descriptors, we are searching for the nearest neighbor in a high-dimensional space. For example, a SIFT descriptor is a vector of 128 numbers. If we have 2,000 keypoints in each image, a Brute-Force matcher would perform 4,000,000 comparisons ($2{,}000 \times 2{,}000$), which is computationally expensive.

To solve this, we use the **Fast Library for Approximate Nearest Neighbors (FLANN)**.

FLANN is a library of algorithms optimized for fast search in large datasets. Instead of checking every single possibility, it uses clever data structures — like trees or hash tables — to quickly narrow down the best candidates.

The "Approximate" in its name is key: it might not always find the absolute closest neighbor, but it finds a "good enough" neighbor significantly faster than a brute-force approach. In the context of image stitching, where we process thousands of points, this trade-off between perfect accuracy and high speed is essential for a responsive pipeline.

---

## Implementing the Matcher and Visualizing

Now, let's build our full `match_descriptors` function step by step. First, we will check our inputs and use our helper function to see if we have binary descriptors.

```python
import cv2
import numpy as np

def match_descriptors(des1, des2, ratio=0.75):
    if des1 is None or des2 is None:
        return []
    if len(des1) < 2 or len(des2) < 2:
        return []

    binary = descriptors_are_binary(des1) or descriptors_are_binary(des2)

```

Next, we need to configure OpenCV's Fast Library for Approximate Nearest Neighbors (FLANN) matcher. This is a highly optimized matcher, but it requires specific configuration dictionaries based on our data type.

```python
    if binary:
        # Settings for binary descriptors (ORB, AKAZE)
        index_params = dict(
            algorithm=6,
            table_number=6,
            key_size=12,
            multi_probe_level=1,
        )
        des1 = np.asarray(des1, dtype=np.uint8)
        des2 = np.asarray(des2, dtype=np.uint8)
    else:
        # Settings for floating-point descriptors (SIFT)
        index_params = dict(algorithm=1, trees=5)
        des1 = np.asarray(des1, dtype=np.float32)
        des2 = np.asarray(des2, dtype=np.float32)

    matcher = cv2.FlannBasedMatcher(index_params, dict(checks=50))

```

Here, we provide the exact algorithm parameters that OpenCV requires to process either binary or floating-point data efficiently. We also ensure our arrays are converted to the correct `dtype` format.

Now, we can ask the matcher for our top two candidates, apply the ratio test loop we learned earlier, and sort the results.

```python
    raw_matches = matcher.knnMatch(des1, des2, k=2)
    good = []
    for pair in raw_matches:
        if len(pair) == 2:
            best, second = pair
            if best.distance < ratio * second.distance:
                good.append(best)

    # Sort the matches so the best ones (shortest distance) are first
    return sorted(good, key=lambda match: match.distance)

```

By returning the matches sorted by `match.distance`, the most confident pairs are always placed at the beginning of our list.

To verify our work, we can use OpenCV's `cv2.drawMatches` function. This takes our two images, their keypoints, and our list of good matches, and draws lines connecting the corresponding features.

```python
# Assuming left and right images, along with their keypoints (kp) and descriptors (des) are ready
matches = match_descriptors(des1, des2, ratio=0.75)

print("method: sift")
print("ratio: 0.75")
print("left keypoints:", len(kp1))
print("right keypoints:", len(kp2))
print("good matches:", len(matches))

preview = cv2.drawMatches(
    left, kp1,
    right, kp2,
    matches[:60], # Draw only the top 60 to avoid clutter
    None,
    flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS,
)

```

When you run this on a pair of images, your output will look something like this:

```text
method: sift
ratio: 0.75
left keypoints: 2000
right keypoints: 2000
good matches: 438

```

The resulting preview image will beautifully display the two photos side by side with colorful lines connecting the exact same locations in both views.

---

## Summary and Next Steps

In this lesson, you took a major step forward in building your image stitcher. You learned how to inspect descriptor data types, configure a FLANN matcher for floating-point or binary descriptors, and apply Lowe's Ratio Test to filter unreliable matches.

This matching process provides the essential links we need. In our next and final unit, we will turn matching into a repeatable diagnostic report so we can warn the user before attempting more fragile geometry steps such as homography estimation.

Now, head over to the upcoming practice exercises. You will write the matching logic yourself and experiment with how different ratio values affect the number and quality of good matches.

## Choosing the Right Feature Detector

Welcome to the start of your matching report journey! Before two images can be matched, you need descriptors from both images. This first exercise quickly rebuilds the detector factory from the previous unit so that the matching code has reliable features to work with.

Open features.py and complete the create_detector function so that it returns a detector based on the method argument.

Here is what to handle, in order:

    For "sift", check whether cv2 has the SIFT_create attribute. If it does not, raise a ValueError with a clear message; otherwise, return cv2.SIFT_create(nfeatures=nfeatures).
    For "orb", return cv2.ORB_create(nfeatures=nfeatures).
    For "akaze", return cv2.AKAZE_create() (no nfeatures argument here).
    For anything else, raise a ValueError whose message includes the unknown method name.

This is a short review step, but it belongs in the matching workflow because matching cannot happen until both images have descriptors.

```
import cv2


def create_detector(method="sift", nfeatures=2000):
    # TODO: If method is "sift", first check that SIFT is available
    # in this OpenCV build using hasattr(cv2, "SIFT_create").
    # If it is not available, raise a ValueError with a clear message.
    # Otherwise, return cv2.SIFT_create(nfeatures=nfeatures).

    # TODO: If method is "orb", return cv2.ORB_create(nfeatures=nfeatures).

    # TODO: If method is "akaze", return cv2.AKAZE_create().
    # Note: AKAZE does NOT accept the nfeatures argument.

    # TODO: If none of the above matched, raise a ValueError that
    # includes the unknown method name in the message.
    pass


if __name__ == "__main__":
    # Quick check: run this file directly to see if your detector is created.
    detector = create_detector()
    print("Created detector:", type(detector).__name__)

```

Here is the completed `create_detector` function in `features.py`:

```python
import cv2


def create_detector(method="sift", nfeatures=2000):
    method = str(method).lower()

    if method == "sift":
        if not hasattr(cv2, "SIFT_create"):
            raise ValueError("SIFT is not available in this OpenCV build")
        return cv2.SIFT_create(nfeatures=nfeatures)

    if method == "orb":
        return cv2.ORB_create(nfeatures=nfeatures)

    if method == "akaze":
        return cv2.AKAZE_create()

    raise ValueError(f"Unknown feature detection method: {method}")


if __name__ == "__main__":
    # Quick check: run this file directly to see if your detector is created.
    detector = create_detector()
    print("Created detector:", type(detector).__name__)

```

## Detecting Keypoints and Computing Descriptors

## Matching Descriptors with Lowes Ratio Test

## Wiring Up the Matching Pipeline

## Tuning the Matching Report Defaults

## Tuning the Matching Report Defaults